# Sprint 1 — Conhecendo os Dados
## Análise Espacial da Criminalidade no Brasil — Sinesp VDE 2015 a 2026

**Aluno(a):** Cintia Nunes  
**Curso:** Ciência da Computação — UFFS

Este notebook foi preparado para trabalhar com **todas as 12 planilhas anuais do Sinesp VDE**, de 2015 a 2026.

Os arquivos verificados possuem a mesma estrutura principal:

- `uf`
- `municipio`
- `evento`
- `data_referencia`
- `agente`
- `arma`
- `faixa_etaria`
- `feminino`
- `masculino`
- `nao_informado`
- `total_vitima`
- `total`
- `total_peso`
- `abrangencia`

### Estratégia utilizada

Como as 12 planilhas juntas possuem milhões de registros, o notebook **não mantém todos os dados brutos simultaneamente na memória**.

Ele:

1. lê cada ano separadamente;
2. mantém somente as colunas necessárias para esta EDA;
3. cria um resumo mensal por UF, indicador e abrangência;
4. libera o arquivo bruto da memória;
5. usa esse resumo para os gráficos, evolução temporal, taxas e mapas;
6. recarrega apenas o ano escolhido quando for necessária a análise por município.

Isso torna a execução mais viável no Google Colab.

## 1. Fonte oficial

**Ministério da Justiça e Segurança Pública — Sinesp VDE**

Base de Dados e Notas Metodológicas dos Gestores Estaduais — Sinesp VDE 2015 a 2026:

https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/dados-nacionais-1/base-de-dados-e-notas-metodologicas-dos-gestores-estaduais-sinesp-vde-2022-e-2023

### Variável utilizada

- `evento` → indicador;
- `data_referencia` → ano e mês;
- `total_vitima` → utilizada quando preenchida;
- `total` → utilizada quando `total_vitima` estiver vazio.

A base possui indicadores contabilizados por vítimas e outros contabilizados por ocorrências. Por isso, o notebook cria a variável `valor` utilizando primeiro `total_vitima` e, quando ela não estiver disponível, `total`.

## 2. Preparação do ambiente

In [ ]:
import sys
import subprocess
import importlib.util

pacotes = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "requests": "requests",
    "geopandas": "geopandas",
    "folium": "folium",
    "ipywidgets": "ipywidgets",
    "python_calamine": "python-calamine",
    "mapclassify": "mapclassify",
    "pyogrio": "pyogrio",
}

faltando = [
    pacote
    for modulo, pacote in pacotes.items()
    if importlib.util.find_spec(modulo) is None
]

if faltando:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *faltando]
    )

print("Ambiente preparado.")

In [ ]:
from pathlib import Path
import gc
import re
import zipfile
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
import ipywidgets as widgets

from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PASTA_DADOS = Path("dados")
PASTA_PROCESSADOS = PASTA_DADOS / "processados"
PASTA_RESULTADOS = Path("resultados")

for pasta in [
    PASTA_DADOS,
    PASTA_PROCESSADOS,
    PASTA_RESULTADOS
]:
    pasta.mkdir(parents=True, exist_ok=True)

print("Pastas preparadas.")

## 3. Carregamento de todas as planilhas

Execute a célula abaixo e selecione **todos os arquivos de 2015 a 2026 de uma só vez**.

Arquivos esperados:

- BancoVDE 2015.xlsx
- BancoVDE 2016.xlsx
- BancoVDE 2017.xlsx
- BancoVDE 2018.xlsx
- BancoVDE 2019.xlsx
- BancoVDE 2020.xlsx
- BancoVDE 2021.xlsx
- BancoVDE 2022.xlsx
- BancoVDE 2023.xlsx
- BancoVDE 2024.xlsx
- BancoVDE 2025.xlsx
- BancoVDE 2026.xlsx

> O nome do arquivo de 2015 também pode conter `(1)` ou `(2)`. O ano é identificado automaticamente.

In [ ]:
USAR_GOOGLE_DRIVE = False
PASTA_GOOGLE_DRIVE = "/content/drive/MyDrive/Sinesp"

if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    caminhos = list(
        Path(PASTA_GOOGLE_DRIVE).glob("BancoVDE*.xlsx")
    )

else:
    from google.colab import files

    print("Selecione todas as planilhas de 2015 a 2026.")
    arquivos_enviados = files.upload()

    caminhos = [
        Path(nome)
        for nome in arquivos_enviados.keys()
    ]

    del arquivos_enviados
    gc.collect()

if not caminhos:
    raise RuntimeError("Nenhuma planilha foi encontrada.")

def ano_do_arquivo(caminho):
    resultado = re.search(
        r"(201[5-9]|202[0-6])",
        caminho.name
    )
    return int(resultado.group(1)) if resultado else None

arquivos_por_ano = {}

for caminho in caminhos:
    ano = ano_do_arquivo(caminho)

    if ano is not None:
        arquivos_por_ano[ano] = caminho

anos_encontrados = sorted(
    arquivos_por_ano.keys()
)

print("Anos encontrados:", anos_encontrados)

esperados = set(range(2015, 2027))
faltando = sorted(
    esperados - set(anos_encontrados)
)

if faltando:
    print("ATENÇÃO — anos não enviados:", faltando)
else:
    print("Todas as 12 planilhas foram identificadas.")

## 4. Funções de preparação

A leitura utiliza `calamine`, que é mais rápida que `openpyxl` para esses arquivos grandes.

A função abaixo lê somente as colunas necessárias para a análise.

In [ ]:
COLUNAS_NECESSARIAS = [
    "uf",
    "municipio",
    "evento",
    "data_referencia",
    "total_vitima",
    "total",
    "abrangencia"
]

MESES = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}

UFS_VALIDAS = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
}

def ler_ano(ano, incluir_municipio=True):
    caminho = arquivos_por_ano[int(ano)]

    planilha = str(ano)

    try:
        excel = pd.ExcelFile(
            caminho,
            engine="calamine"
        )

        if planilha not in excel.sheet_names:
            planilha = excel.sheet_names[0]

        colunas = COLUNAS_NECESSARIAS.copy()

        if not incluir_municipio:
            colunas.remove("municipio")

        df = pd.read_excel(
            caminho,
            sheet_name=planilha,
            usecols=colunas,
            engine="calamine"
        )

    except Exception as erro:
        raise RuntimeError(
            f"Erro ao ler {caminho.name}: {erro}"
        )

    df["data_referencia"] = pd.to_datetime(
        df["data_referencia"],
        errors="coerce"
    )

    df["ano"] = df["data_referencia"].dt.year
    df["mes_numero"] = df["data_referencia"].dt.month
    df["mes"] = df["mes_numero"].map(MESES)

    df["uf"] = (
        df["uf"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df["indicador"] = (
        df["evento"]
        .astype(str)
        .str.strip()
    )

    if incluir_municipio:
        df["municipio"] = (
            df["municipio"]
            .astype(str)
            .str.strip()
        )

    df["abrangencia"] = (
        df["abrangencia"]
        .fillna("Não informado")
        .astype(str)
        .str.strip()
    )

    total_vitima = pd.to_numeric(
        df["total_vitima"],
        errors="coerce"
    )

    total = pd.to_numeric(
        df["total"],
        errors="coerce"
    )

    df["valor"] = total_vitima.combine_first(total)

    df = df.dropna(
        subset=[
            "uf",
            "indicador",
            "ano",
            "mes_numero",
            "valor"
        ]
    ).copy()

    df["ano"] = df["ano"].astype(int)
    df["mes_numero"] = df["mes_numero"].astype(int)

    df = df[
        df["uf"].isin(UFS_VALIDAS)
    ].copy()

    return df

## 5. Processamento das 12 planilhas

Esta é a etapa mais demorada do notebook.

Cada planilha é carregada **uma única vez**, resumida e removida da memória.

O resultado final é uma base muito menor contendo:

- ano;
- mês;
- UF;
- indicador;
- abrangência;
- valor total.

In [ ]:
resumos = []
cobertura = []
qualidade = []

for ano in anos_encontrados:
    print(f"Processando {ano}...")

    df_ano = ler_ano(
        ano,
        incluir_municipio=False
    )

    cobertura.append({
        "ano": ano,
        "arquivo": arquivos_por_ano[ano].name,
        "registros_brutos": len(df_ano),
        "mes_inicial": int(df_ano["mes_numero"].min()),
        "mes_final": int(df_ano["mes_numero"].max()),
        "indicadores": int(df_ano["indicador"].nunique()),
        "ufs": int(df_ano["uf"].nunique()),
        "abrangencias": int(df_ano["abrangencia"].nunique())
    })

    qualidade.append({
        "ano": ano,
        "valores_zero": int((df_ano["valor"] == 0).sum()),
        "valores_negativos": int((df_ano["valor"] < 0).sum()),
        "uf_unicas": int(df_ano["uf"].nunique()),
        "indicadores_unicos": int(df_ano["indicador"].nunique())
    })

    resumo_ano = (
        df_ano
        .groupby(
            [
                "ano",
                "mes_numero",
                "mes",
                "uf",
                "indicador",
                "abrangencia"
            ],
            as_index=False,
            dropna=False
        )["valor"]
        .sum()
    )

    resumos.append(
        resumo_ano
    )

    del df_ano
    del resumo_ano
    gc.collect()

resumo_sinesp = pd.concat(
    resumos,
    ignore_index=True
)

cobertura = pd.DataFrame(
    cobertura
).sort_values("ano")

qualidade = pd.DataFrame(
    qualidade
).sort_values("ano")

del resumos
gc.collect()

print()
print("Processamento concluído.")
print("Dimensão do resumo:", resumo_sinesp.shape)

## 6. Cobertura temporal das planilhas

In [ ]:
display(cobertura)

cobertura["cobertura_meses"] = (
    cobertura["mes_final"]
    .apply(lambda x: MESES.get(x, x))
)

parciais = cobertura[
    cobertura["mes_final"] < 12
]

if not parciais.empty:
    print("ATENÇÃO: existem anos com cobertura parcial:")
    display(
        parciais[
            [
                "ano",
                "mes_inicial",
                "mes_final",
                "cobertura_meses"
            ]
        ]
    )

## 7. Qualidade dos dados

In [ ]:
display(qualidade)

print(
    "Registros no resumo:",
    len(resumo_sinesp)
)

print(
    "Anos:",
    sorted(resumo_sinesp["ano"].unique())
)

print(
    "Quantidade total de indicadores diferentes:",
    resumo_sinesp["indicador"].nunique()
)

## 8. Lista de todos os indicadores

Aqui são mostrados **todos os indicadores encontrados nas planilhas**, sem limitar aos 10 maiores.

In [ ]:
indicadores_gerais = (
    resumo_sinesp["indicador"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
    .to_frame("indicador")
)

display(indicadores_gerais)

print(
    "Total de indicadores diferentes:",
    len(indicadores_gerais)
)

## 9. Escolha do ano, indicador e abrangência

O usuário escolhe:

- o ano;
- o indicador;
- a abrangência.

Quando o ano muda, a lista de indicadores é atualizada automaticamente para mostrar somente os indicadores existentes naquele ano.

In [ ]:
anos_disponiveis = sorted(
    resumo_sinesp["ano"]
    .unique()
    .tolist()
)

seletor_ano = widgets.Dropdown(
    options=anos_disponiveis,
    value=max(anos_disponiveis),
    description="Ano:"
)

def indicadores_do_ano(ano):
    return sorted(
        resumo_sinesp.loc[
            resumo_sinesp["ano"] == int(ano),
            "indicador"
        ]
        .dropna()
        .unique()
        .tolist()
    )

indicadores_iniciais = indicadores_do_ano(
    seletor_ano.value
)

seletor_indicador = widgets.Dropdown(
    options=indicadores_iniciais,
    value=indicadores_iniciais[0],
    description="Indicador:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="750px")
)

def abrangencias_do_ano(ano):
    valores = sorted(
        resumo_sinesp.loc[
            resumo_sinesp["ano"] == int(ano),
            "abrangencia"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    return ["Todas"] + valores

abrangencias_iniciais = abrangencias_do_ano(
    seletor_ano.value
)

seletor_abrangencia = widgets.Dropdown(
    options=abrangencias_iniciais,
    value="Todas",
    description="Abrangência:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="500px")
)

botao_confirmar = widgets.Button(
    description="Confirmar seleção",
    button_style="success"
)

saida_selecao = widgets.Output()

ANO_ANALISE = int(
    seletor_ano.value
)

INDICADOR_ANALISE = (
    seletor_indicador.value
)

ABRANGENCIA_ANALISE = (
    seletor_abrangencia.value
)

def atualizar_opcoes(change):
    ano = int(change["new"])

    indicadores = indicadores_do_ano(
        ano
    )

    seletor_indicador.options = (
        indicadores
    )

    if indicadores:
        seletor_indicador.value = (
            indicadores[0]
        )

    abrangencias = abrangencias_do_ano(
        ano
    )

    seletor_abrangencia.options = (
        abrangencias
    )

    seletor_abrangencia.value = (
        "Todas"
    )

def confirmar_selecao(_):
    global ANO_ANALISE
    global INDICADOR_ANALISE
    global ABRANGENCIA_ANALISE

    ANO_ANALISE = int(
        seletor_ano.value
    )

    INDICADOR_ANALISE = (
        seletor_indicador.value
    )

    ABRANGENCIA_ANALISE = (
        seletor_abrangencia.value
    )

    with saida_selecao:
        clear_output()

        print("Seleção confirmada.")
        print("Ano:", ANO_ANALISE)
        print("Indicador:", INDICADOR_ANALISE)
        print("Abrangência:", ABRANGENCIA_ANALISE)
        print()
        print(
            "Agora execute as células abaixo "
            "para atualizar a análise."
        )

seletor_ano.observe(
    atualizar_opcoes,
    names="value"
)

botao_confirmar.on_click(
    confirmar_selecao
)

display(
    widgets.VBox(
        [
            seletor_ano,
            seletor_indicador,
            seletor_abrangencia,
            botao_confirmar,
            saida_selecao
        ]
    )
)

## 10. Todos os indicadores do ano selecionado

Este gráfico apresenta **todos os indicadores disponíveis no ano escolhido**.

Não existe corte de Top 10.

In [ ]:
dados_ano = resumo_sinesp[
    resumo_sinesp["ano"] == ANO_ANALISE
].copy()

if ABRANGENCIA_ANALISE != "Todas":
    dados_ano = dados_ano[
        dados_ano["abrangencia"]
        == ABRANGENCIA_ANALISE
    ].copy()

totais_indicadores_ano = (
    dados_ano
    .groupby(
        "indicador",
        as_index=False
    )["valor"]
    .sum()
    .sort_values(
        "valor",
        ascending=False
    )
)

display(
    totais_indicadores_ano
)

dados_grafico = (
    totais_indicadores_ano
    .sort_values(
        "valor",
        ascending=True
    )
)

altura = max(
    8,
    len(dados_grafico) * 0.38
)

ax = dados_grafico.plot(
    x="indicador",
    y="valor",
    kind="barh",
    figsize=(13, altura),
    legend=False
)

ax.set_title(
    f"Todos os indicadores — {ANO_ANALISE}"
)

ax.set_xlabel(
    "Total registrado"
)

ax.set_ylabel(
    "Indicador"
)

plt.tight_layout()
plt.show()

## 11. Evolução do indicador selecionado entre 2015 e 2026

In [ ]:
serie_historica = resumo_sinesp[
    resumo_sinesp["indicador"]
    == INDICADOR_ANALISE
].copy()

if ABRANGENCIA_ANALISE != "Todas":
    serie_historica = serie_historica[
        serie_historica["abrangencia"]
        == ABRANGENCIA_ANALISE
    ].copy()

serie_historica = (
    serie_historica
    .groupby(
        "ano",
        as_index=False
    )["valor"]
    .sum()
    .sort_values("ano")
)

display(
    serie_historica
)

ax = serie_historica.plot(
    x="ano",
    y="valor",
    kind="line",
    marker="o",
    figsize=(11, 5),
    legend=False
)

ax.set_title(
    f"Evolução histórica — {INDICADOR_ANALISE}"
)

ax.set_xlabel("Ano")
ax.set_ylabel("Total registrado")

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

anos_parciais = cobertura.loc[
    cobertura["mes_final"] < 12,
    "ano"
].tolist()

anos_parciais_na_serie = [
    ano
    for ano in serie_historica["ano"].tolist()
    if ano in anos_parciais
]

if anos_parciais_na_serie:
    print(
        "Atenção: a série contém ano(s) com cobertura parcial:",
        anos_parciais_na_serie
    )

## 12. Distribuição mensal do indicador selecionado

In [ ]:
recorte = resumo_sinesp[
    (resumo_sinesp["ano"] == ANO_ANALISE)
    &
    (
        resumo_sinesp["indicador"]
        == INDICADOR_ANALISE
    )
].copy()

if ABRANGENCIA_ANALISE != "Todas":
    recorte = recorte[
        recorte["abrangencia"]
        == ABRANGENCIA_ANALISE
    ].copy()

if recorte.empty:
    raise RuntimeError(
        "Não existem registros para a seleção realizada."
    )

por_mes = (
    recorte
    .groupby(
        [
            "mes_numero",
            "mes"
        ],
        as_index=False
    )["valor"]
    .sum()
    .sort_values(
        "mes_numero"
    )
)

display(por_mes)

ax = por_mes.plot(
    x="mes",
    y="valor",
    kind="line",
    marker="o",
    figsize=(11, 5),
    legend=False
)

ax.set_title(
    f"Evolução mensal — {INDICADOR_ANALISE}, {ANO_ANALISE}"
)

ax.set_xlabel("Mês")
ax.set_ylabel("Total registrado")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Análise por Unidade da Federação

In [ ]:
por_uf = (
    recorte
    .groupby(
        "uf",
        as_index=False
    )["valor"]
    .sum()
    .rename(
        columns={
            "valor":
            "ocorrencias"
        }
    )
    .sort_values(
        "ocorrencias",
        ascending=False
    )
)

display(por_uf)

ax = (
    por_uf
    .sort_values("ocorrencias")
    .plot(
        x="uf",
        y="ocorrencias",
        kind="barh",
        figsize=(10, 8),
        legend=False
    )
)

ax.set_title(
    f"{INDICADOR_ANALISE} por UF — {ANO_ANALISE}"
)

ax.set_xlabel(
    "Total registrado"
)

ax.set_ylabel("UF")

plt.tight_layout()
plt.show()

## 14. Possíveis outliers entre as UFs

In [ ]:
q1 = por_uf[
    "ocorrencias"
].quantile(0.25)

q3 = por_uf[
    "ocorrencias"
].quantile(0.75)

iqr = q3 - q1

limite_inferior = (
    q1 - 1.5 * iqr
)

limite_superior = (
    q3 + 1.5 * iqr
)

por_uf["outlier_iqr"] = (
    (
        por_uf["ocorrencias"]
        < limite_inferior
    )
    |
    (
        por_uf["ocorrencias"]
        > limite_superior
    )
)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print(
    "Limite superior:",
    limite_superior
)

display(
    por_uf[
        por_uf["outlier_iqr"]
    ]
    .sort_values(
        "ocorrencias",
        ascending=False
    )
)

ax = por_uf[
    "ocorrencias"
].plot(
    kind="box",
    figsize=(5, 6)
)

ax.set_title(
    f"Distribuição entre UFs — {INDICADOR_ANALISE}, {ANO_ANALISE}"
)

ax.set_ylabel(
    "Total registrado"
)

plt.tight_layout()
plt.show()

## 15. Análise por município

Para economizar memória, nesta etapa o notebook recarrega **somente a planilha do ano selecionado**.

O gráfico mostra os 30 municípios com maior valor apenas para facilitar a visualização. A tabela `por_municipio` contém **todos os municípios**.

In [ ]:
print(
    f"Carregando dados municipais de {ANO_ANALISE}..."
)

df_municipios = ler_ano(
    ANO_ANALISE,
    incluir_municipio=True
)

print(
    "Registros carregados:",
    len(df_municipios)
)

print(
    "\nIndicadores disponíveis:"
)

display(
    sorted(
        df_municipios[
            "indicador"
        ]
        .dropna()
        .unique()
    )
)

df_municipios = df_municipios[
    df_municipios["indicador"]
    == INDICADOR_ANALISE
].copy()

print(
    "\nRegistros após filtro do indicador:",
    len(df_municipios)
)

if ABRANGENCIA_ANALISE != "Todas":

    df_municipios = df_municipios[
        df_municipios["abrangencia"]
        == ABRANGENCIA_ANALISE
    ].copy()

    print(
        "Registros após filtro de abrangência:",
        len(df_municipios)
    )

municipios_invalidos = {
    "",
    "NAN",
    "NÃO INFORMADO",
    "NAO INFORMADO"
}

df_municipios["municipio"] = (
    df_municipios["municipio"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_municipios = df_municipios[
    ~df_municipios[
        "municipio"
    ]
    .str.upper()
    .isin(
        municipios_invalidos
    )
].copy()

por_municipio = (
    df_municipios
    .groupby(
        [
            "uf",
            "municipio"
        ],
        as_index=False
    )["valor"]
    .sum()
    .rename(
        columns={
            "valor":
            "ocorrencias"
        }
    )
    .sort_values(
        "ocorrencias",
        ascending=False
    )
)

print(
    "\nMunicípios encontrados:",
    len(por_municipio)
)

if por_municipio.empty:

    print(
        "\nNão existem dados municipais "
        "para esta seleção."
    )

    print(
        "Ano:",
        ANO_ANALISE
    )

    print(
        "Indicador:",
        INDICADOR_ANALISE
    )

    print(
        "Abrangência:",
        ABRANGENCIA_ANALISE
    )

else:

    display(
        por_municipio
    )

    top_municipios = (
        por_municipio
        .head(30)
        .sort_values(
            "ocorrencias"
        )
    )

    top_municipios[
        "municipio_uf"
    ] = (
        top_municipios[
            "municipio"
        ]
        + " - "
        + top_municipios[
            "uf"
        ]
    )

    ax = top_municipios.plot(
        x="municipio_uf",
        y="ocorrencias",
        kind="barh",
        figsize=(11, 10),
        legend=False
    )

    ax.set_title(
        f"30 municípios com maior total — "
        f"{INDICADOR_ANALISE}, "
        f"{ANO_ANALISE}"
    )

    ax.set_xlabel(
        "Total registrado"
    )

    ax.set_ylabel(
        "Município"
    )

    plt.tight_layout()
    plt.show()

del df_municipios

gc.collect()

## 16. População e taxa por 100 mil habitantes

O notebook tenta consultar a população correspondente ao ano escolhido no SIDRA/IBGE.

Quando a população não estiver disponível para aquele ano nessa tabela, a análise continua utilizando valores absolutos.

In [ ]:
SIDRA_TABELA_POP = 6579
SIDRA_VARIAVEL_POP = 9324

CODIGO_UF = {
    11: "RO",
    12: "AC",
    13: "AM",
    14: "RR",
    15: "PA",
    16: "AP",
    17: "TO",
    21: "MA",
    22: "PI",
    23: "CE",
    24: "RN",
    25: "PB",
    26: "PE",
    27: "AL",
    28: "SE",
    29: "BA",
    31: "MG",
    32: "ES",
    33: "RJ",
    35: "SP",
    41: "PR",
    42: "SC",
    43: "RS",
    50: "MS",
    51: "MT",
    52: "GO",
    53: "DF"
}

def carregar_populacao_uf(ano):
    colunas = [
        "uf",
        "codigo_uf",
        "nome_uf",
        "ano",
        "populacao"
    ]

    url = (
        "https://apisidra.ibge.gov.br/values/"
        f"t/{SIDRA_TABELA_POP}/n3/all/"
        f"v/{SIDRA_VARIAVEL_POP}/p/{ano}"
    )

    try:
        resposta = requests.get(
            url,
            timeout=60
        )

        resposta.raise_for_status()
        dados = resposta.json()

    except Exception as erro:
        print(
            "Não foi possível consultar a população:",
            erro
        )
        return pd.DataFrame(columns=colunas)

    if not dados or len(dados) < 2:
        return pd.DataFrame(columns=colunas)

    df = pd.DataFrame(dados)

    if str(
        df.iloc[0].get("V", "")
    ).lower() == "valor":
        df = df.iloc[1:].copy()

    if not {
        "D1C",
        "D1N",
        "V"
    }.issubset(df.columns):
        return pd.DataFrame(columns=colunas)

    pop = df[
        [
            "D1C",
            "D1N",
            "V"
        ]
    ].copy()

    pop.columns = [
        "codigo_uf",
        "nome_uf",
        "populacao"
    ]

    pop["codigo_uf"] = pd.to_numeric(
        pop["codigo_uf"],
        errors="coerce"
    )

    pop["populacao"] = pd.to_numeric(
        pop["populacao"],
        errors="coerce"
    )

    pop = pop.dropna(
        subset=[
            "codigo_uf",
            "populacao"
        ]
    )

    pop["codigo_uf"] = (
        pop["codigo_uf"]
        .astype(int)
    )

    pop["uf"] = (
        pop["codigo_uf"]
        .map(CODIGO_UF)
    )

    pop["ano"] = int(ano)

    return pop[colunas]


pop_uf = carregar_populacao_uf(
    ANO_ANALISE
)

if pop_uf.empty:
    print(
        f"População não disponível nesta consulta para {ANO_ANALISE}."
    )
else:
    display(
        pop_uf.head()
    )

In [ ]:
analise_uf = por_uf.drop(
    columns=["outlier_iqr"],
    errors="ignore"
).copy()

if not pop_uf.empty:
    analise_uf = analise_uf.merge(
        pop_uf,
        on="uf",
        how="left"
    )

    analise_uf["taxa_100_mil"] = (
        analise_uf["ocorrencias"]
        /
        analise_uf["populacao"]
        *
        100000
    )

else:
    analise_uf["codigo_uf"] = np.nan
    analise_uf["nome_uf"] = np.nan
    analise_uf["ano"] = ANO_ANALISE
    analise_uf["populacao"] = np.nan
    analise_uf["taxa_100_mil"] = np.nan

if analise_uf[
    "taxa_100_mil"
].notna().any():

    analise_uf = (
        analise_uf
        .sort_values(
            "taxa_100_mil",
            ascending=False
        )
    )

    display(
        analise_uf[
            [
                "uf",
                "ocorrencias",
                "populacao",
                "taxa_100_mil"
            ]
        ].round(
            {
                "taxa_100_mil":
                2
            }
        )
    )

    ax = (
        analise_uf
        .sort_values(
            "taxa_100_mil"
        )
        .plot(
            x="uf",
            y="taxa_100_mil",
            kind="barh",
            figsize=(10, 8),
            legend=False
        )
    )

    ax.set_title(
        f"Taxa por 100 mil habitantes — "
        f"{INDICADOR_ANALISE}, {ANO_ANALISE}"
    )

    ax.set_xlabel(
        "Taxa por 100 mil habitantes"
    )

    ax.set_ylabel("UF")

    plt.tight_layout()
    plt.show()

else:
    print(
        "Taxa por 100 mil habitantes não calculada "
        "para este ano."
    )

    display(
        analise_uf[
            [
                "uf",
                "ocorrencias"
            ]
        ]
    )

## 17. Mapa temático do Brasil

In [ ]:
URL_MALHA_UF = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_municipais/municipio_2024/"
    "Brasil/BR_UF_2024.zip"
)

arquivo_zip = Path(
    "BR_UF_2024.zip"
)

pasta_malha = Path(
    "BR_UF_2024"
)

if not arquivo_zip.exists():
    print(
        "Baixando malha do IBGE..."
    )

    resposta = requests.get(
        URL_MALHA_UF,
        timeout=180
    )

    resposta.raise_for_status()

    arquivo_zip.write_bytes(
        resposta.content
    )

if not pasta_malha.exists():
    pasta_malha.mkdir(
        exist_ok=True
    )

    with zipfile.ZipFile(
        arquivo_zip,
        "r"
    ) as z:
        z.extractall(
            pasta_malha
        )

shapefiles = list(
    pasta_malha.rglob(
        "*.shp"
    )
)

if not shapefiles:
    raise FileNotFoundError(
        "Shapefile do IBGE não encontrado."
    )

ufs_geo = gpd.read_file(
    shapefiles[0]
)

ufs_geo.columns = [
    str(c).lower()
    for c in ufs_geo.columns
]

if "sigla_uf" in ufs_geo.columns:
    ufs_geo = ufs_geo.rename(
        columns={
            "sigla_uf":
            "uf"
        }
    )

elif "uf" not in ufs_geo.columns:
    if "cd_uf" in ufs_geo.columns:
        ufs_geo["uf"] = (
            pd.to_numeric(
                ufs_geo["cd_uf"],
                errors="coerce"
            )
            .map(CODIGO_UF)
        )
    else:
        raise RuntimeError(
            "Não foi possível localizar a UF na malha."
        )

mapa_uf = ufs_geo.merge(
    analise_uf[
        [
            "uf",
            "ocorrencias",
            "populacao",
            "taxa_100_mil"
        ]
    ],
    on="uf",
    how="left"
)

if mapa_uf[
    "taxa_100_mil"
].notna().any():

    VARIAVEL_MAPA = (
        "taxa_100_mil"
    )

    TITULO_MAPA = (
        "taxa por 100 mil habitantes"
    )

else:
    VARIAVEL_MAPA = (
        "ocorrencias"
    )

    TITULO_MAPA = (
        "total registrado"
    )

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 10)
)

quantidade_classes = min(
    5,
    mapa_uf[
        VARIAVEL_MAPA
    ]
    .dropna()
    .nunique()
)

opcoes = {
    "column":
    VARIAVEL_MAPA,
    "legend":
    True,
    "edgecolor":
    "white",
    "linewidth":
    0.5,
    "ax":
    ax,
    "missing_kwds":
    {
        "label":
        "Sem dados"
    }
}

if quantidade_classes >= 2:
    opcoes[
        "scheme"
    ] = "quantiles"

    opcoes[
        "k"
    ] = quantidade_classes

mapa_uf.plot(
    **opcoes
)

ax.set_title(
    f"{INDICADOR_ANALISE} — "
    f"{TITULO_MAPA} ({ANO_ANALISE})",
    fontsize=14
)

ax.axis("off")

plt.tight_layout()

arquivo_mapa = (
    PASTA_RESULTADOS
    / "mapa_analise_uf.png"
)

plt.savefig(
    arquivo_mapa,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(
    "Mapa salvo em:",
    arquivo_mapa
)

## 18. Mapa interativo

In [ ]:
mapa_wgs84 = mapa_uf.to_crs(
    epsg=4326
).copy()

mapa_interativo = folium.Map(
    location=[
        -14.5,
        -52.5
    ],
    zoom_start=4,
    tiles="CartoDB positron"
)

folium.Choropleth(
    geo_data=mapa_wgs84,
    data=mapa_wgs84,
    columns=[
        "uf",
        VARIAVEL_MAPA
    ],
    key_on="feature.properties.uf",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name=(
        f"{TITULO_MAPA.capitalize()} — "
        f"{INDICADOR_ANALISE} ({ANO_ANALISE})"
    )
).add_to(
    mapa_interativo
)

campos = [
    "uf",
    "ocorrencias"
]

aliases = [
    "UF:",
    "Total registrado:"
]

if mapa_wgs84[
    "populacao"
].notna().any():
    campos.append(
        "populacao"
    )
    aliases.append(
        "População:"
    )

if mapa_wgs84[
    "taxa_100_mil"
].notna().any():
    campos.append(
        "taxa_100_mil"
    )
    aliases.append(
        "Taxa por 100 mil:"
    )

folium.GeoJson(
    mapa_wgs84,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "weight": 0.5
    },
    tooltip=folium.GeoJsonTooltip(
        fields=campos,
        aliases=aliases,
        localize=True
    )
).add_to(
    mapa_interativo
)

arquivo_html = (
    PASTA_RESULTADOS
    / "mapa_interativo.html"
)

mapa_interativo.save(
    arquivo_html
)

print(
    "Mapa interativo salvo em:",
    arquivo_html
)

mapa_interativo

## 19. Exportação dos dados processados

In [ ]:
arquivo_resumo = (
    PASTA_PROCESSADOS
    / "sinesp_resumo_2015_2026.csv"
)

arquivo_cobertura = (
    PASTA_PROCESSADOS
    / "cobertura_2015_2026.csv"
)

arquivo_indicadores = (
    PASTA_PROCESSADOS
    / "indicadores.csv"
)

arquivo_uf = (
    PASTA_PROCESSADOS
    / "analise_uf_selecionada.csv"
)

arquivo_municipios = (
    PASTA_PROCESSADOS
    / "analise_municipios_selecionada.csv"
)

resumo_sinesp.to_csv(
    arquivo_resumo,
    index=False,
    encoding="utf-8-sig"
)

cobertura.to_csv(
    arquivo_cobertura,
    index=False,
    encoding="utf-8-sig"
)

indicadores_gerais.to_csv(
    arquivo_indicadores,
    index=False,
    encoding="utf-8-sig"
)

analise_uf.to_csv(
    arquivo_uf,
    index=False,
    encoding="utf-8-sig"
)

por_municipio.to_csv(
    arquivo_municipios,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos gerados:")
print("-", arquivo_resumo)
print("-", arquivo_cobertura)
print("-", arquivo_indicadores)
print("-", arquivo_uf)
print("-", arquivo_municipios)
print("-", PASTA_RESULTADOS / "mapa_analise_uf.png")
print("-", PASTA_RESULTADOS / "mapa_interativo.html")

## 20. Conclusões da EDA

Após executar o notebook, utilize os resultados para registrar:

1. quais anos estão disponíveis e até qual mês cada base possui registros;
2. quantos indicadores diferentes aparecem no período;
3. quais indicadores possuem maiores totais no ano selecionado;
4. como o indicador escolhido evoluiu entre 2015 e 2026;
5. quais UFs apresentam maiores registros;
6. quais municípios apresentam maior concentração;
7. se existem possíveis outliers;
8. como o ranking muda quando são utilizadas taxas por 100 mil habitantes;
9. quais limitações foram encontradas nos dados.

### Atenção à interpretação de 2026

Se a tabela de cobertura mostrar que 2026 possui menos de 12 meses, o ano deve ser tratado como **parcial** e não deve ser comparado diretamente a anos completos sem registrar essa limitação.

## 21. Limitações

- Os arquivos anuais possuem milhões de registros e podem levar alguns minutos para serem processados.
- A disponibilidade e a forma de registro podem variar entre as Unidades da Federação.
- Indicadores diferentes podem utilizar `total_vitima` ou `total`.
- A variável `abrangencia` deve ser considerada durante a interpretação.
- Anos com menos de 12 meses são identificados como parciais.
- A taxa por 100 mil habitantes só é calculada quando a consulta populacional do IBGE retorna dados para o ano selecionado.
- A análise é exploratória e não estabelece relação causal para a criminalidade.